In [1]:
import aether.config as config 
import numpy as np
import cupy as cp
import aether as ae

In [2]:
from pathlib import Path
# Navigate 2 levels up from 'examples/pure_grit/' to the repository root
REPO_ROOT = Path(__file__).resolve().parents[2] if "__file__" in locals() else Path.cwd().parents[1]
DATA_PATH = REPO_ROOT / "data" / "cifar-10-python.tar.gz"

print(REPO_ROOT)

print(f"\n{DATA_PATH}")

/app

/app/data/cifar-10-python.tar.gz


In [3]:
TRAIN_DIR = REPO_ROOT / "data" / "cifar-10" / "cifar-10_train.npz"
TEST_DIR  = REPO_ROOT / "data" / "cifar-10" / "cifar-10_test.npz"

with cp.load(TRAIN_DIR, allow_pickle=False) as data:
    X_train = data["X_train"]
    y_train = data["y_train"]

with cp.load(TEST_DIR, allow_pickle=False) as data:
    X_test = data["X_test"]
    y_test = data["y_test"]


Now lets do our preprocessing pipeline

In [4]:
TARGET_DEVICE = "cupy"
feature_pipeline = ae.Compose([
    ae.ToTensor(dtype='float32', target_device=TARGET_DEVICE),
    ae.Rescale(factor=1.0 / 255.0),
    ae.StandardScaler()
]).fit(X_train)

# 2. Transform train and test features seamlessly
X_train_tensor = feature_pipeline(X_train)
X_test_tensor = feature_pipeline(X_test)

# 3. Convert target labels
y_train_tensor, y_test_tensor = ae.to_tensor(
    y_train, y_test, target_device=TARGET_DEVICE, preserve_integers=True
)

print(f"{X_train.shape=}, {X_train.dtype=}, {type(X_train)=}")
print(f"{y_train.shape=}, {y_train.dtype=}, {type(y_train)}")
print(f"{X_train_tensor.shape=}, {X_train_tensor.dtype=}, {type(X_train_tensor)=}")
print(f"{y_train_tensor.shape=}, {y_train_tensor.dtype=}, {type(y_train_tensor)}")


X_train.shape=(50000, 32, 32, 3), X_train.dtype=dtype('uint8'), type(X_train)=<class 'cupy.ndarray'>
y_train.shape=(50000,), y_train.dtype=dtype('int64'), <class 'cupy.ndarray'>
X_train_tensor.shape=(50000, 32, 32, 3), X_train_tensor.dtype=dtype('float32'), type(X_train_tensor)=<class 'cupy.ndarray'>
y_train_tensor.shape=(50000,), y_train_tensor.dtype=dtype('int64'), <class 'cupy.ndarray'>


## Making a Simple CNN Model that Uses Half Precision

These models aren't meant to be serious contendors for benchmark performance, but rather a proof of concept. Second, Conv2d kernel is quite slow at the moment, so I am working on a fix for $3\times3$ filters with $1\times1$ stride for better performance.

In [12]:
model = ae.Model()
model.add(ae.Conv2d(3, 32, (3,3), (1,1), padding="same"))
model.add(ae.MaxPool2d((2,2), (2,2), padding="valid"))
model.add(ae.ReLU())
model.add(ae.Conv2d(32, 64, (3,3), (1,1), padding="same"))
model.add(ae.LeakyReLU(alpha=0.01))
model.add(ae.SpatialDropout(rate=0.05, seed=42))
model.add(ae.GlobalAvgPool())
model.add(ae.Dense(64, 10))
model.configure(
    loss = ae.SoftmaxCategoricalCrossEntropy(label_smoothing=0.01),
    optimizer= ae.Adam(learning_rate=0.001, decay=5e-5),
    accuracy= ae.CategoricalAccuracy()
)
#model.set_precision("float16")
model.to('cupy')
model.set_precision(compute_dtype="float16")
model.finalize(input_shape = X_train_tensor.shape[1:])
model.train(
    X=X_train_tensor, 
    y=y_train_tensor, 
    epochs=5, 
    batch_size=128,
    print_every=150, 
    validation_data=(X_test_tensor, y_test_tensor)
)

Epoch 1/5 | Step 1/391 - loss: 3.2269 (data: 3.2269, reg: 0.0000) - acc: 0.1172 - lr: 0.001000
Epoch 1/5 | Step 151/391 - loss: 2.0421 (data: 2.0421, reg: 0.0000) - acc: 0.2422 - lr: 0.000993
Epoch 1/5 | Step 301/391 - loss: 1.8408 (data: 1.8408, reg: 0.0000) - acc: 0.3203 - lr: 0.000985
Epoch 1/5 | Step 391/391 - loss: 1.8902 (data: 1.8902, reg: 0.0000) - acc: 0.3250 - lr: 0.000981
[Epoch 1/5 Total] loss: 1.9866 - acc: 0.2785 - lr: 0.000981
[Validation] loss: 1.8041 - acc: 0.3314
Epoch 2/5 | Step 1/391 - loss: 2.0037 (data: 2.0037, reg: 0.0000) - acc: 0.2500 - lr: 0.000981
Epoch 2/5 | Step 151/391 - loss: 1.7566 (data: 1.7566, reg: 0.0000) - acc: 0.3594 - lr: 0.000974
Epoch 2/5 | Step 301/391 - loss: 1.6901 (data: 1.6901, reg: 0.0000) - acc: 0.4062 - lr: 0.000967
Epoch 2/5 | Step 391/391 - loss: 1.7816 (data: 1.7816, reg: 0.0000) - acc: 0.3375 - lr: 0.000962
[Epoch 2/5 Total] loss: 1.7554 - acc: 0.3622 - lr: 0.000962
[Validation] loss: 1.6439 - acc: 0.4028
Epoch 3/5 | Step 1/391 - los

In [ ]:
model = ae.Model()
model.add(ae.Flatten())
model.add(ae.Dense(32*32*3, 128))
model.add(ae.ReLU())
model.add(ae.Dense(128, 10))
model.configure(
    loss = ae.SoftmaxCategoricalCrossEntropy(label_smoothing=0.01),
    optimizer= ae.Adam(learning_rate=0.001, decay=5e-5),
    accuracy= ae.CategoricalAccuracy()
)
#model.set_precision("float16") works here too
model.to('cupy')
model.set_precision(compute_dtype="float16")
model.finalize(input_shape=X_train_tensor.shape[1:])
model.train(
    X=X_train_tensor, 
    y=y_train_tensor, 
    epochs=5, 
    batch_size=128,
    print_every=150, 
    validation_data=(X_test_tensor, y_test_tensor)
)

Epoch 1/5 | Step 1/391 - loss: 3.0339 (data: 3.0339, reg: 0.0000) - acc: 0.1094 - lr: 0.001000
Epoch 1/5 | Step 151/391 - loss: 1.7857 (data: 1.7857, reg: 0.0000) - acc: 0.4141 - lr: 0.000993
Epoch 1/5 | Step 301/391 - loss: 1.8139 (data: 1.8139, reg: 0.0000) - acc: 0.3672 - lr: 0.000985
Epoch 1/5 | Step 391/391 - loss: 1.8252 (data: 1.8252, reg: 0.0000) - acc: 0.4625 - lr: 0.000981
[Epoch 1/5 Total] loss: 1.8920 - acc: 0.3918 - lr: 0.000981
[Validation] loss: 1.6498 - acc: 0.4436
Epoch 2/5 | Step 1/391 - loss: 1.4174 (data: 1.4174, reg: 0.0000) - acc: 0.5234 - lr: 0.000981
Epoch 2/5 | Step 151/391 - loss: 1.5775 (data: 1.5775, reg: 0.0000) - acc: 0.4766 - lr: 0.000974
Epoch 2/5 | Step 301/391 - loss: 1.5199 (data: 1.5199, reg: 0.0000) - acc: 0.5000 - lr: 0.000967
Epoch 2/5 | Step 391/391 - loss: 1.5758 (data: 1.5758, reg: 0.0000) - acc: 0.4750 - lr: 0.000962
[Epoch 2/5 Total] loss: 1.5556 - acc: 0.4803 - lr: 0.000962
[Validation] loss: 1.5371 - acc: 0.4752
Epoch 3/5 | Step 1/391 - los